In [1]:
# =============================================================================
# CELL 1 — Imports
# =============================================================================

from __future__ import annotations

import importlib
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

import src.pago_pipeline.ncbi_snapshot as ncbi_snapshot_module

# Reload the pipeline module so notebook reruns pick up local code changes.
ncbi_snapshot_module = importlib.reload(ncbi_snapshot_module)

SnapshotMode = ncbi_snapshot_module.SnapshotMode
get_snapshot_xml_file_path = ncbi_snapshot_module.get_snapshot_xml_file_path
resolve_ncbi_protein_uid_snapshot = (
    ncbi_snapshot_module.resolve_ncbi_protein_uid_snapshot
)
resolve_ncbi_protein_xml_snapshot = (
    ncbi_snapshot_module.resolve_ncbi_protein_xml_snapshot
)

from src.pago_pipeline.storage import sha256_of_file, sha256_of_lines

In [2]:
# =============================================================================
# CELL 2 — Load environment and resolve project root
# =============================================================================
dotenv_path = find_dotenv(usecwd=False)

if not dotenv_path:
    raise FileNotFoundError(
        "Could not find a .env file while walking up parent directories. "
        "Place .env with your NCBI email and optional API key at the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

PROJECT_ROOT = Path(dotenv_path).resolve().parent
NCBI_EMAIL = os.getenv("NCBI_EMAIL")
NCBI_API_KEY = os.getenv("NCBI_API_KEY")

if not NCBI_EMAIL:
    raise ValueError(
        "NCBI_EMAIL was not found in the environment. "
        "Please define it in your .env file."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"NCBI email configured: {bool(NCBI_EMAIL)}")
print(f"NCBI API key configured: {bool(NCBI_API_KEY)}")

Project root: C:\Programming\Python\pAgo-project
NCBI email configured: True
NCBI API key configured: True


In [3]:
# =============================================================================
# CELL 3 — Define XML snapshot configuration
# =============================================================================

# Options: "local_excel" or "latest_uid_snapshot".
PROTEIN_IDENTIFIER_SOURCE = "local_excel"
LOCAL_EXCEL_PROTEIN_IDENTIFIER_FILE_PATH = (
    PROJECT_ROOT
    / "data"
    / "01-raw"
    / "Protein_acession_version"
    / "mbo006184236st1.xls"
)
LOCAL_EXCEL_SHEET_NAME = "novel_pago_fixed_july2018"
LOCAL_EXCEL_IDENTIFIER_COLUMN = "Accession"

SOURCE_UID_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "01-raw"
    / "protein_uid_snapshots"
    / "local_excel_mbo006184236st1"
)
XML_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "01-raw" / "protein_xml_snapshots"
)

SOURCE_UID_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create
XML_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create
MAX_RETRY_ATTEMPTS = 5

DEDUPLICATE_IDENTIFIERS = True
SORT_IDENTIFIERS = True

XML_BATCH_SIZE = 50
XML_REQUEST_DELAY_SECONDS = None
SOCKET_IDLE_TIMEOUT_SECONDS = 90.0
BATCH_DEADLINE_SECONDS = 300.0
CIRCUIT_BREAKER_FAILURE_THRESHOLD = 3
CIRCUIT_BREAKER_COOLDOWN_SECONDS = 180.0

UPDATE_LATEST_DIRECTORY = True

print(f"Protein identifier source: {PROTEIN_IDENTIFIER_SOURCE}")
print(f"Local Excel protein identifier file: {LOCAL_EXCEL_PROTEIN_IDENTIFIER_FILE_PATH}")
print(f"Local Excel sheet name: {LOCAL_EXCEL_SHEET_NAME}")
print(f"Local Excel identifier column: {LOCAL_EXCEL_IDENTIFIER_COLUMN}")
print(f"Source UID snapshot root directory: {SOURCE_UID_SNAPSHOT_ROOT_DIRECTORY}")
print(f"XML snapshot root directory: {XML_SNAPSHOT_ROOT_DIRECTORY}")
print(f"Source UID snapshot mode: {SOURCE_UID_SNAPSHOT_MODE}")
print(f"XML snapshot mode: {XML_SNAPSHOT_MODE}")
print(f"XML batch size: {XML_BATCH_SIZE}")
print(f"Socket idle timeout seconds: {SOCKET_IDLE_TIMEOUT_SECONDS}")
print(f"Batch deadline seconds: {BATCH_DEADLINE_SECONDS}")

Protein identifier source: local_excel
Local Excel protein identifier file: C:\Programming\Python\pAgo-project\data\01-raw\Protein_acession_version\mbo006184236st1.xls
Local Excel sheet name: novel_pago_fixed_july2018
Local Excel identifier column: Accession
Source UID snapshot root directory: C:\Programming\Python\pAgo-project\data\01-raw\protein_uid_snapshots\local_excel_mbo006184236st1
XML snapshot root directory: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots
Source UID snapshot mode: reuse_latest_or_create
XML snapshot mode: reuse_latest_or_create
XML batch size: 50
Socket idle timeout seconds: 90.0
Batch deadline seconds: 300.0


In [4]:
# =============================================================================
# CELL 4 — Resolve active XML snapshot
# =============================================================================

if PROTEIN_IDENTIFIER_SOURCE == "local_excel":
    source_uid_snapshot_payload = resolve_ncbi_protein_uid_snapshot(
        snapshot_mode=SOURCE_UID_SNAPSHOT_MODE,
        snapshot_root_directory=SOURCE_UID_SNAPSHOT_ROOT_DIRECTORY,
        local_excel_file_path=LOCAL_EXCEL_PROTEIN_IDENTIFIER_FILE_PATH,
        local_excel_sheet_name=LOCAL_EXCEL_SHEET_NAME,
        local_excel_identifier_column=LOCAL_EXCEL_IDENTIFIER_COLUMN,
        deduplicate_uids=DEDUPLICATE_IDENTIFIERS,
        sort_uids=SORT_IDENTIFIERS,
        update_latest_directory=UPDATE_LATEST_DIRECTORY,
    )
elif PROTEIN_IDENTIFIER_SOURCE == "latest_uid_snapshot":
    source_uid_snapshot_payload = resolve_ncbi_protein_uid_snapshot(
        snapshot_mode=SnapshotMode.reuse_latest,
        snapshot_root_directory=SOURCE_UID_SNAPSHOT_ROOT_DIRECTORY,
    )
else:
    raise ValueError(
        "Invalid PROTEIN_IDENTIFIER_SOURCE. Expected 'local_excel' or "
        "'latest_uid_snapshot'."
    )

xml_snapshot_payload = resolve_ncbi_protein_xml_snapshot(
    snapshot_mode=XML_SNAPSHOT_MODE,
    snapshot_root_directory=XML_SNAPSHOT_ROOT_DIRECTORY,
    source_uid_snapshot_root_directory=SOURCE_UID_SNAPSHOT_ROOT_DIRECTORY,
    source_uid_snapshot_payload=source_uid_snapshot_payload,
    xml_batch_size=XML_BATCH_SIZE,
    max_retry_attempts=MAX_RETRY_ATTEMPTS,
    xml_request_delay_seconds=XML_REQUEST_DELAY_SECONDS,
    socket_idle_timeout_seconds=SOCKET_IDLE_TIMEOUT_SECONDS,
    batch_deadline_seconds=BATCH_DEADLINE_SECONDS,
    circuit_breaker_failure_threshold=CIRCUIT_BREAKER_FAILURE_THRESHOLD,
    circuit_breaker_cooldown_seconds=CIRCUIT_BREAKER_COOLDOWN_SECONDS,
    ncbi_email=NCBI_EMAIL,
    ncbi_api_key=NCBI_API_KEY,
    update_latest_directory=UPDATE_LATEST_DIRECTORY,
)

xml_snapshot_directory = xml_snapshot_payload["snapshot_directory"]
manifest_file_path = xml_snapshot_payload["manifest_file_path"]
protein_uids_file_path = xml_snapshot_payload["protein_uids_file_path"]
xml_snapshot_manifest = xml_snapshot_payload["manifest"]
protein_uids = xml_snapshot_payload["protein_uids"]

xml_file_path = get_snapshot_xml_file_path(
    snapshot_directory=xml_snapshot_directory,
)
uid_dataset_sha256 = sha256_of_lines(
    text_lines=protein_uids,
    deduplicate_lines_preserving_order=False,
    sort_lines=False,
)

print(f"Resolved XML snapshot directory: {xml_snapshot_directory}")
print(f"Resolved XML UID count: {len(protein_uids)}")
print(f"Resolved XML batch count: {xml_snapshot_manifest['batch_count']}")

Latest snapshot is available. Reusing frozen snapshot.
Latest XML snapshot is available but does not match the requested source UID snapshot. Creating a new frozen XML snapshot.
Starting XML request for batch 1/21 (attempt 1/5) with 50 protein UIDs.
Received XML response for batch 1/21 in 6.533 seconds (537213 bytes, 115209.4 bytes/s).
NCBI XML request URL: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=protein&id=NP_213999.1%2CNP_443795.1%2CNP_630911.1%2CNP_681738.1%2CNP_952414.1%2CWP_000151321.1%2CWP_002566035.1%2CWP_002645493.1%2CWP_002801241.1%2CWP_002993156.1%2CWP_004290502.1%2CWP_004308218.1%2CWP_004318406.1%2CWP_004328663.1%2CWP_004492459.1%2CWP_004594459.1%2CWP_004624807.1%2CWP_004672358.1%2CWP_004710421.1%2CWP_004748891.1%2CWP_004968197.1%2CWP_004992799.1%2CWP_005032630.1%2CWP_005277191.1%2CWP_005372460.1%2CWP_005580376.1%2CWP_005783349.1%2CWP_005801097.1%2CWP_005864495.1%2CWP_005988487.1%2CWP_006019180.1%2CWP_006054116.1%2CWP_006090832.1%2CWP_006111085.1%2CWP_00

In [5]:
# =============================================================================
# CELL 5 — Print XML snapshot summary
# =============================================================================

xml_file_sha256 = sha256_of_file(input_file_path=xml_file_path)

print("XML snapshot resolved successfully.")
print(f"Immutable XML snapshot directory: {xml_snapshot_directory}")
print(f"Consolidated XML file: {xml_file_path}")
print(f"Consolidated XML SHA-256: {xml_file_sha256}")
print(f"Total protein UIDs covered: {len(protein_uids)}")
print(f"Total XML batches used for consolidation: {xml_snapshot_manifest['batch_count']}")

XML snapshot resolved successfully.
Immutable XML snapshot directory: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots\snapshots\2026-06-01T20-43-40Z__q_345f747e4e5b
Consolidated XML file: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots\snapshots\2026-06-01T20-43-40Z__q_345f747e4e5b\protein_records.xml
Consolidated XML SHA-256: 6034ed027f360827fea6c555715c435fa727cec6592111f6b9f7eab77e997d6a
Total protein UIDs covered: 1010
Total XML batches used for consolidation: 21


In [6]:
# =============================================================================
# CELL 6 — Print manifest-derived retrieval summary
# =============================================================================

print(f"Retrieved at UTC: {xml_snapshot_manifest['retrieved_at_utc']}")
print(f"Manifest batch size: {xml_snapshot_manifest['batch_size']}")
print(f"Manifest batch count: {xml_snapshot_manifest['batch_count']}")
print(
    "Manifest normalized protein UID count: "
    f"{xml_snapshot_manifest['normalized_protein_uid_count']}"
)
print(
    "Manifest consolidated record count: "
    f"{xml_snapshot_manifest['consolidated_record_count']}"
)
print(
    "Source UID snapshot relative path: "
    f"{xml_snapshot_manifest['source_uid_snapshot_relative_path']}"
)

Retrieved at UTC: 2026-06-01T20:43:40Z
Manifest batch size: 50
Manifest batch count: 21
Manifest normalized protein UID count: 1010
Manifest consolidated record count: 1010
Source UID snapshot relative path: snapshots\2026-06-01T20-41-44Z__q_345f747e4e5b


In [7]:
# =============================================================================
# CELL 7 — Print persisted file hashes
# =============================================================================

saved_manifest_sha256 = sha256_of_file(input_file_path=manifest_file_path)
saved_protein_uids_file_sha256 = sha256_of_file(
    input_file_path=protein_uids_file_path,
)

print("Persisted XML snapshot file hashes:")
print(f"Manifest SHA-256: {saved_manifest_sha256}")
print(f"Protein UIDs file SHA-256: {saved_protein_uids_file_sha256}")
print(f"Consolidated XML file SHA-256: {xml_file_sha256}")

Persisted XML snapshot file hashes:
Manifest SHA-256: 948b0a1493af5816be54749b10be83900667ed03f04e5c8f7d86a785e003604c
Protein UIDs file SHA-256: 7423cd908138b93606026489213d3293920cb7583d0e0019094de20686c5d52f
Consolidated XML file SHA-256: 6034ed027f360827fea6c555715c435fa727cec6592111f6b9f7eab77e997d6a


In [8]:
# =============================================================================
# CELL 8 — Print persisted manifest summary
# =============================================================================

print("Persisted XML snapshot metadata:")
print(f"Manifest path: {manifest_file_path}")
print(f"Protein UIDs file path: {protein_uids_file_path}")
print(f"Manifest XML file name: {xml_snapshot_manifest['xml_file_name']}")
print(f"Manifest XML SHA-256: {xml_snapshot_manifest['xml_file_sha256']}")
print(
    "Manifest immutable snapshot relative path: "
    f"{xml_snapshot_manifest['immutable_snapshot_relative_path']}"
)
print(
    "Manifest source UID snapshot relative path: "
    f"{xml_snapshot_manifest['source_uid_snapshot_relative_path']}"
)

Persisted XML snapshot metadata:
Manifest path: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots\snapshots\2026-06-01T20-43-40Z__q_345f747e4e5b\manifest.json
Protein UIDs file path: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots\snapshots\2026-06-01T20-43-40Z__q_345f747e4e5b\protein_uids.txt
Manifest XML file name: protein_records.xml
Manifest XML SHA-256: 6034ed027f360827fea6c555715c435fa727cec6592111f6b9f7eab77e997d6a
Manifest immutable snapshot relative path: snapshots\2026-06-01T20-43-40Z__q_345f747e4e5b
Manifest source UID snapshot relative path: snapshots\2026-06-01T20-41-44Z__q_345f747e4e5b


In [9]:
# =============================================================================
# CELL 9 — Inspect first saved batch records
# =============================================================================

first_three_saved_batch_records = xml_snapshot_manifest["batches"][:3]

for saved_batch_record in first_three_saved_batch_records:
    print(f"Batch index: {saved_batch_record['batch_index']}")
    print(
        "Batch UID interval: "
        f"{saved_batch_record['batch_start_index']}"
        f"..{saved_batch_record['batch_end_index']}"
    )
    print(saved_batch_record["xml_payload_sha256"])
    print(saved_batch_record["protein_uid_count"])
    print("---")

Batch index: 1
Batch UID interval: 0..49
47fdaf2321857347f6cfc9f549d8fb91284a84063d8d580ce49926d630132ac1
50
---
Batch index: 2
Batch UID interval: 50..99
3856efa20e4a8d98f38510bf7abb043e17556f67be06e6edc0a0dd0eb0a8876f
50
---
Batch index: 3
Batch UID interval: 100..149
cd6b814c08dbd3db67ac63e932587fdf597d1cd1e39e8c5f393a8af2340eff28
50
---


In [10]:
# =============================================================================
# CELL 10 — Validate persisted snapshot consistency
# =============================================================================

print("XML snapshot resolution completed successfully.")
print(f"Immutable XML snapshot directory: {xml_snapshot_directory}")
print(f"Consolidated XML file: {xml_file_path}")
print(f"Protein UIDs file: {protein_uids_file_path}")
print(f"Manifest file: {manifest_file_path}")
print(
    "Manifest XML file name matches saved path: "
    f"{xml_snapshot_manifest['xml_file_name'] == xml_file_path.name}"
)
print(
    "Manifest XML SHA-256 matches saved XML file: "
    f"{xml_snapshot_manifest['xml_file_sha256'] == xml_file_sha256}"
)
print(
    "Manifest UID SHA-256 matches resolved UID list: "
    f"{xml_snapshot_manifest['protein_uids_sha256'] == uid_dataset_sha256}"
)
print(
    "Manifest UID count matches resolved UID list: "
    f"{xml_snapshot_manifest['normalized_protein_uid_count'] == len(protein_uids)}"
)

XML snapshot resolution completed successfully.
Immutable XML snapshot directory: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots\snapshots\2026-06-01T20-43-40Z__q_345f747e4e5b
Consolidated XML file: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots\snapshots\2026-06-01T20-43-40Z__q_345f747e4e5b\protein_records.xml
Protein UIDs file: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots\snapshots\2026-06-01T20-43-40Z__q_345f747e4e5b\protein_uids.txt
Manifest file: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots\snapshots\2026-06-01T20-43-40Z__q_345f747e4e5b\manifest.json
Manifest XML file name matches saved path: True
Manifest XML SHA-256 matches saved XML file: True
Manifest UID SHA-256 matches resolved UID list: True
Manifest UID count matches resolved UID list: True


In [11]:
# =============================================================================
# CELL 11 — Print source UID snapshot provenance
# =============================================================================
print("Source UID snapshot provenance:")
print(
    "Source UID snapshot relative path: "
    f"{xml_snapshot_manifest['source_uid_snapshot_relative_path']}"
)
print(
    "Source UID snapshot directory name: "
    f"{xml_snapshot_manifest['source_uid_snapshot_directory_name']}"
)
print(
    "Source UID snapshot manifest SHA-256: "
    f"{xml_snapshot_manifest['source_uid_snapshot_manifest_sha256']}"
)
print(f"Source UID count: {xml_snapshot_manifest['source_uid_count']}")

Source UID snapshot provenance:
Source UID snapshot relative path: snapshots\2026-06-01T20-41-44Z__q_345f747e4e5b
Source UID snapshot directory name: 2026-06-01T20-41-44Z__q_345f747e4e5b
Source UID snapshot manifest SHA-256: e998dda6b98812fa8f7c5004eb95a7c540213501ca9927b3f391198cc58d6b75
Source UID count: 1010
